In [13]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from core import *
from utils import *
from lark import Tree, Token
from pm4py import save_vis_petri_net
import pandas as pd
import sys
#print(f"Versione Python: {sys.version}")

#print(torch.cuda.is_available())
#print(torch.__version__)

# SETTINGS
NARY = 1
PROBABILITIES = 0.25,0.25,0.25,0.25
FILE_PATH_PNG = "petri_net_output.png"
TRACE_ENC_REG = "data/regions_full.csv"
TRACE_ENC_TAS = "data/tasks_full.csv"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
learning_rate = 3e-4

cuda


In [14]:
'''def get_batch(split, train_data_X, train_data_Y, val_data_X, val_data_Y, batch_size, device):
    # Seleziona i dati corretti
    X_data = train_data_X if split == 'train' else val_data_X
    Y_data = train_data_Y if split == 'train' else val_data_Y

    # Genera indici casuali
    ix = torch.randint(len(X_data), (batch_size,))

    # Estrae e sposta sul device
    x = X_data[ix].to(device)
    y = Y_data[ix].to(device)

    return x, y'''

def get_batch(split, train_data, val_data, batch_size, block_size, device):
    # Seleziona i dati corretti
    data = train_data if split == 'train' else val_data

    # Genera indici casuali
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device):
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            # Richiama get_batch passando i parametri ricevuti
            X, Y = get_batch(split, train_data, val_data, batch_size, block_size, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [15]:
iterations = 3  # Quante task diverse
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

# ALBERO GIOCATTOLO
# Da rimuovere
'''tree = Tree('xor', [
    # 1. Primo figlio di XOR (T1)
    Tree('task', [Token('NAME', 'T1')]),

    # 2. Secondo figlio di XOR (Blocco Sequential)
    Tree('sequential', [

        # 2.1 Primo figlio di Sequential (Parallel)
        Tree('parallel', [
            # 2.1.1 Ramo sinistro del Parallel (Xor annidati)
            Tree('xor', [
                Tree('task', [Token('NAME', 'T2')]),
                Tree('xor', [
                    Tree('task', [Token('NAME', 'T3')]),
                    Tree('task', [Token('NAME', 'T4')])
                ])
            ]),
            # 2.1.2 Ramo destro del Parallel (Parallel T5, T6)
            Tree('parallel', [
                Tree('task', [Token('NAME', 'T5')]),
                Tree('task', [Token('NAME', 'T6')])
            ])
        ]),

        # 2.2 Secondo figlio di Sequential (Sequential T7, T8)
        Tree('sequential', [
            Tree('task', [Token('NAME', 'T7')]),
            Tree('task', [Token('NAME', 'T8')])
        ])
    ])
])'''

tree = Tree('xor', [Tree('loop', [Tree('sequential', [Tree('loop', [Tree('task', [Token('NAME', 'T1')])]), Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')])])])]), Tree('parallel', [Tree('task', [Token('NAME', 'T4')]), Tree('loop', [Tree('task', [Token('NAME', 'T5')])])])])

if NARY:
    tree = createNAryTree(tree)
    print(tree)

Tree('xor', [Tree('loop', [Tree('sequential', [Tree('loop', [Tree('task', [Token('NAME', 'T1')])]), Tree('xor', [Tree('task', [Token('NAME', 'T2')]), Tree('task', [Token('NAME', 'T3')])])])]), Tree('parallel', [Tree('task', [Token('NAME', 'T4')]), Tree('loop', [Tree('task', [Token('NAME', 'T5')])])])])


In [16]:
# Oggetto PetriNetP - si inizializza automaticamente con il suo costruttore
net = PetriNetP(tree)

save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    FILE_PATH_PNG,
    format="png"  # Specifica il formato
)

# Oggetto Generator
generator = Generator(300, net)

In [17]:
# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

traceEncoded_regions, traceEncoded_tasks = getEncoding(generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses)

num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

traceEncoded_regions.to_csv(TRACE_ENC_REG, index=True)
traceEncoded_tasks.to_csv(TRACE_ENC_TAS, index=True)

df_traces = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(df_traces)

df_traces = df_traces.T

df_tracescopy = df_traces.copy()

unique_columns = df_traces.drop_duplicates()
unique_tuple = [tuple(x) for x in unique_columns.values]

# 2. Creiamo i dizionari di mappatura
# bit_to_id: trasforma la colonna di 12 bit in un numero
# id_to_bit: trasforma il numero nei 12 bit originali (per la generazione)
bit_to_id = {v: i for i, v in enumerate(unique_tuple)}
id_to_bit = {i: v for i, v in enumerate(unique_tuple)}

vocab_size = len(unique_columns)

encode = lambda a: [bit_to_id[tuple(x)] for x in a]
decode = lambda b: [id_to_bit[x] for x in b]

data = torch.tensor(encode(df_traces.values), dtype=torch.long)

#traces = cutTraces(traceEncoded_regions, traceEncoded_tasks)

block_size = 32
n_embd = 64
dropout = 0.35
n_head = 4
n_layer = 3
batch_size = 32
eval_iters = 200
eval_interval = 500
max_iters = 3000

#X, Y = create_training_set(traces,8)

model = BPMNTransformer(vocab_size, num_regions+num_tasks, block_size, n_embd, dropout, n_head, n_layer)
m = model.to(device)

print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

n = int(0.8 * len(df_traces))

train_data = data[:n]
val_data = data[n:]

#train_data_X = X[:n]
#train_data_Y = Y[:n]
#val_data_X = X[n:]
#val_data_Y = Y[n:]

'''xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)
print(xb)
print(yb)'''

    X  +  ->  <>
R0  1  0   0   0
R1  0  0   0   1
R2  0  0   1   0
R3  0  0   0   1
R4  1  0   0   0
R5  0  1   0   0
R6  0  0   0   1
    R0  R1  R2  R3  R4  R5  R6  T1  T2  T3  T4  T5
R0   0   1   0   0   0   1   0   0   0   0   0   0
R1   0   0   1   0   0   0   0   0   0   0   0   0
R2   0   0   0   1   1   0   0   0   0   0   0   0
R3   0   0   0   0   0   0   0   1   0   0   0   0
R4   0   0   0   0   0   0   0   0   1   1   0   0
R5   0   0   0   0   0   0   1   0   0   0   1   0
R6   0   0   0   0   0   0   0   0   0   0   0   1
    0     1     2     3     4     5     6     7     8     9     ...  2756  \
R0     1     1     1     1     1     0     1     1     1     1  ...     1   
R1     1     1     1     1     1     0     0     0     0     0  ...     1   
R2     1     1     1     1     1     0     0     0     0     0  ...     1   
R3     1     1     1     0     0     0     0     0     0     0  ...     0   
R4     0     0     0     0     1     0     0     0     0     0  ...    

"xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)\nprint(xb)\nprint(yb)"

In [18]:
# Dentro il ciclo for iter in range(max_iters):
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # A. Pesca i dati (usando la funzione esterna)
    xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)

    # B. FORWARD PASS: Il modello "gira" e produce una previsione
    logits, loss = model(xb, yb)

    # C. RESET GRADIENTI: Puliamo i calcoli del giro precedente
    optimizer.zero_grad(set_to_none=True)

    # D. BACKWARD PASS: Calcoliamo l'errore per ogni neurone (L'Anima del training)
    loss.backward()

    # E. OPTIMIZER STEP: Aggiorniamo i pesi per sbagliare meno al prossimo giro
    optimizer.step()

step 0: train loss 2.4934, val loss 2.4958
step 500: train loss 0.5337, val loss 0.5155
step 1000: train loss 0.5112, val loss 0.4963
step 1500: train loss 0.5036, val loss 0.4915
step 2000: train loss 0.4994, val loss 0.5000
step 2500: train loss 0.4928, val loss 0.5014
step 2999: train loss 0.4912, val loss 0.5079


In [19]:
context = torch.tensor(encode([[0]*(num_regions+num_tasks)]), dtype=torch.long, device=device).unsqueeze(0)
print(context)
# Generiamo gli ID
generated_indices = m.generate(idx=context, block_size=block_size, max_new_tokens=100)[0].tolist()

# Decodifichiamo in bit
decoded_output = decode(generated_indices)

print("\n--- TRACCE GENERATE ---")
traces_generated = []
current_trace_generated = []
for i, step in enumerate(decoded_output):
    bit_list = [int(b) for b in step]
    current_trace_generated.append(bit_list)
    if bit_list == [0]*(num_regions+num_tasks):
        traces_generated.append(current_trace_generated)
        current_trace_generated = []
    #print(f"Step {i:02d}: {bit_list}")

traces_generated.remove(traces_generated[0]) #Rimuovo la prima che è sempre [], generata ed inserita dall'algoritmo sopra
for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")


tensor([[4]], device='cuda:0')

--- TRACCE GENERATE ---
0: [[1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
1: [[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0], [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
2: [[1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
3: [[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0], [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 

In [20]:
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

# Facciamo il decoding delle tracce generate
#print(traces_generated)
traces_decoded = getDecoding(traces_generated, net.regions, net.tasks)
for i,trace in enumerate(traces_decoded):
    print(f"{i}: {trace}")

# Creiamo il l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)


check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])

silent_transition_prefixes = start + end + loop

model_cost_function = dict()
sync_cost_function = dict()

# METTIAMO 10000 come di default???
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is not None and t.label.startswith(silent_transition_prefixes): # Se è una transizione silente,
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    elif t.label is None:
        model_cost_function[t] = 0
        sync_cost_function[t] = 10000

    else: # Se è un task vero e proprio
        model_cost_function[t] = 10000
        sync_cost_function[t] = 0

parameters = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost_function,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost_function
}

# Eseguiamo l'allineamento
aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=parameters)
for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

0: ['start_T5', 'end_T5', 'start_T4', 'end_T4']
1: ['start_T4', 'end_T4', 'start_T5', 'end_T5']
2: ['start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T2', 'end_T2', 'start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T3', 'end_T3']
3: ['start_T4', 'end_T4', 'start_T5', 'end_T5', 'start_T5', 'end_T5']
4: ['start_T4', 'end_T4', 'start_T5', 'end_T5']
5: ['start_T4', 'end_T4', 'start_T5', 'end_T5']
6: ['start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T3', 'end_T3', 'start_T1', 'end_T1', 'start_T3', 'end_T3', 'start_T1', 'end_T1', 'start_T3', 'end_T3']
7: ['start_T4', 'end_T4', 'start_T5', 'end_T5', 'start_T5', 'end_T5']
8: ['start_T5', 'end_T5', 'start_T4', 'end_T4']
9: ['start_T1', 'end_T1', 'start_T3', 'end_T3', 'start_T1', 'end_T1', 'start_T3', 'end_T3']
10: ['start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T2', 'end_T2']
11: ['start_T5', 'end_T5', 'start_T4', 'end_T4']
12: ['start_T1', 'end_T1', 'start_T1', 'end_T1', 'start_T1', 'end_T1',

aligning log, completed variants :: 100%|██████████| 9/9 [00:00<00:00, 580.11it/s]

0: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P5'), ('>>', 'start_L6'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_L6'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_P5'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 12, 'queued_states': 36, 'traversed_arcs': 36, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 80000}
1: {'alignment': [('>>', 'start_X0'), ('>>', 'start_P5'), ('start_T4', 'start_T4'), ('>>', 'start_L6'), ('end_T4', 'end_T4'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_L6'), ('>>', 'end_P5'), ('>>', 'end_X0')], 'cost': 0, 'visited_states': 12, 'queued_states': 33, 'traversed_arcs': 33, 'lp_solved': 1, 'fitness': 1.0, 'bwc': 80000}
2: {'alignment': [('>>', 'start_X0'), ('>>', 'start_L1'), ('>>', 'start_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'back_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'end_L3'), ('>>', 'start_X4'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('>>', 'end_X4'), (

In [21]:
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
back_loop = tuple(["back_L"])

# Puliamo l'allineamento e otteniamo la traccia allineata più vicina a quella generata
aligned_traces_cleaned = []
for a_trace in aligned_traces:
    trace_cleaned = []
    trace = a_trace['alignment']

    print(trace)

    for _, net_step in trace: #trans --> trans step , net --> real petri net step
        if net_step != None:
            if not (net_step.startswith(start) or net_step.startswith(end) or net_step.startswith(back_loop)) and not net_step=='>>':
                trace_cleaned.append(net_step)

    aligned_traces_cleaned.append(trace_cleaned)


for i,trace in enumerate(aligned_traces_cleaned):
    print(f"{i}: {trace}")

# Codifichiamo le tracce allineati (per poi poterle confrontare con quelle generate dal transformer)
aligned_traceEncoded_regions, aligned_traceEncoded_tasks = getEncoding(aligned_traces_cleaned, net.regions, net.tasks, net.open_clauses, net.end_clauses)
df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)
print(df_aligned_traces)


[('>>', 'start_X0'), ('>>', 'start_P5'), ('>>', 'start_L6'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_L6'), ('start_T4', 'start_T4'), ('end_T4', 'end_T4'), ('>>', 'end_P5'), ('>>', 'end_X0')]
[('>>', 'start_X0'), ('>>', 'start_P5'), ('start_T4', 'start_T4'), ('>>', 'start_L6'), ('end_T4', 'end_T4'), ('start_T5', 'start_T5'), ('end_T5', 'end_T5'), ('>>', 'end_L6'), ('>>', 'end_P5'), ('>>', 'end_X0')]
[('>>', 'start_X0'), ('>>', 'start_L1'), ('>>', 'start_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'back_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'end_L3'), ('>>', 'start_X4'), ('start_T2', 'start_T2'), ('end_T2', 'end_T2'), ('>>', 'end_X4'), ('>>', 'back_L1'), ('>>', 'start_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'back_L3'), ('start_T1', 'start_T1'), ('end_T1', 'end_T1'), ('>>', 'end_L3'), ('>>', 'start_X4'), ('start_T3', 'start_T3'), ('end_T3', 'end_T3'), ('>>', 'end_X4'), ('>>', 'end_L1'), ('>>', 'end_X0')]
[('>

In [22]:
# Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza
aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
for i in range(len(aligned_traces_encoded)):
    generated_t = traces_generated[i]
    aligned_t = aligned_traces_encoded[i]

    gen_tuples = [tuple(step) for step in generated_t]
    aligned_tuples = [tuple(step) for step in aligned_t]

    total_possible_column = set(gen_tuples + aligned_tuples) # Prendo tutte le possibili colonne per andare a creare il dizionario delle sostituzioni

    # Se volessi dizionario delle distanze bisognerebbe usare questo pezzo di codice (adesso usiamo distanza di hamming di base nel codice)
    '''cost_sub = {}
    for e1 in total_possible_column:
        for e2 in total_possible_column:
            if e1 != e2:
                counter = 0
                for i in range(len(e1)):
                    if e1[i] != e2[i]:
                        counter+=1
                cost_sub[(e1,e2)] = counter'''

    cost = edit_distance_weighted_levenshtein(generated_t, aligned_t, num_regions+num_tasks, num_regions+num_tasks, hamming_distance) # Utilizziamo la distanza di hamming al momento
    print(cost)

#costo = edit_distance_weighted_levenshtein(traccia_ai, traccia_pulita, cost_ins, cost_del, cost_sub)
#print(f"Costo di correzione totale: {costo}")

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
